In [20]:
from pyspark.sql import SparkSession

# 1. Inicialização da Sessão Spark
spark = SparkSession.builder \
    .appName("Telecom-PDD-Gold-ResumoFinanceiro") \
    .getOrCreate()

gold_path = "s3://data-lake-elt-303004020922-us-east-1-an/projeto-pdd/03-refined/"


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [21]:
# Lendo a tabela detalhada de Aging List gerada no passo anterior
df_aging_list=spark.read.parquet(f"{gold_path}aging_list_detalhada/")

df_aging_list.createOrReplaceTempView("df_aging_list")


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [22]:
df_aging_list.show(5)

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

+----------+----------+-----------+------------+---------------+----------+-----------+--------+---------+---------+-----------+
| fatura_id|cliente_id|      plano|valor_fatura|data_vencimento| data_proc|dias_atraso|flg_1_30|flg_31_60|flg_61_90|flg_90_mais|
+----------+----------+-----------+------------+---------------+----------+-----------+--------+---------+---------+-----------+
|FAT_000229| CLI_00482|Fibra 500MB|      129.90|     2024-02-13|2026-03-31|        777|       0|        0|        0|          1|
|FAT_000315| CLI_00245|Fibra 300MB|       99.90|     2024-02-07|2026-03-31|        783|       0|        0|        0|          1|
|FAT_000420| CLI_00594|Fibra 300MB|       99.90|     2024-02-24|2026-03-31|        766|       0|        0|        0|          1|
|FAT_000848| CLI_00125|Fibra 300MB|       99.90|     2024-02-01|2026-03-31|        789|       0|        0|        0|          1|
|FAT_000910| CLI_00842|Fibra 500MB|      129.90|     2024-02-21|2026-03-31|        769|       0| 

In [23]:
# 2. Processamento Gold: Agregação por Faixa de Risco e Aplicação das Taxas de PDD
# Definimos taxas gerenciais (simuladas) de telecomunicações para provisão:
# - 1 a 30 dias: 2% de risco
# - 31 a 60 dias: 10% de risco
# - 61 a 90 dias: 50% de risco
# - Mais de 90 dias: 100% de risco (calote provável)

query_resumo_pdd = """
    WITH consolidado_faixas AS (
        SELECT 
            '1 a 30 dias' as faixa_atraso,
            SUM(valor_fatura) as valor_total_aberto,
            COUNT(fatura_id) as qtd_faturas,
            0.02 as taxa_pdd
        FROM df_aging_list
        WHERE flg_1_30 = 1
        
        UNION ALL
        
        SELECT 
            '31 a 60 dias' as faixa_atraso,
            SUM(valor_fatura) as valor_total_aberto,
            COUNT(fatura_id) as qtd_faturas,
            0.10 as taxa_pdd
        FROM df_aging_list
        WHERE flg_31_60 = 1
        
        UNION ALL
        
        SELECT 
            '61 a 90 dias' as faixa_atraso,
            SUM(valor_fatura) as valor_total_aberto,
            COUNT(fatura_id) as qtd_faturas,
            0.50 as taxa_pdd
        FROM df_aging_list
        WHERE flg_61_90 = 1
        
        UNION ALL
        
        SELECT 
            'Mais de 90 dias' as faixa_atraso,
            SUM(valor_fatura) as valor_total_aberto,
            COUNT(fatura_id) as qtd_faturas,
            1.00 as taxa_pdd
        FROM df_aging_list
        WHERE flg_90_mais = 1
    )
    SELECT 
        faixa_atraso,
        COALESCE(qtd_faturas, 0) as qtd_faturas,
        COALESCE(valor_total_aberto, 0.0) as valor_total_aberto,
        taxa_pdd,
        ROUND(COALESCE(valor_total_aberto, 0.0) * taxa_pdd, 2) as valor_pdd_calculado
    FROM consolidado_faixas
"""

# Executando a query de resumo
df_resumo_pdd = spark.sql(query_resumo_pdd)

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [24]:
df_resumo_pdd.show()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

+---------------+-----------+------------------+--------+-------------------+
|   faixa_atraso|qtd_faturas|valor_total_aberto|taxa_pdd|valor_pdd_calculado|
+---------------+-----------+------------------+--------+-------------------+
|    1 a 30 dias|          1|            129.90|    0.02|               2.60|
|   31 a 60 dias|          3|            289.70|    0.10|              28.97|
|   61 a 90 dias|         22|           2067.80|    0.50|            1033.90|
|Mais de 90 dias|       1476|         151052.40|    1.00|          151052.40|
+---------------+-----------+------------------+--------+-------------------+

In [25]:
# 3. Salvando a tabela de resumo financeiro na Camada Gold
df_resumo_pdd.write \
    .mode("overwrite") \
    .parquet(f"{gold_path}resumo_pdd_mensal/")

print("Tabela Gold 'resumo_pdd_mensal' gerada e salva com sucesso!")

# Registrando view para consulta rápida
df_resumo_pdd.createOrReplaceTempView("tb_gold_resumo_pdd")

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

Tabela Gold 'resumo_pdd_mensal' gerada e salva com sucesso!

In [26]:
# 4. Exibindo o Resultado Executivo do PDD
print("\n Relatorio Executivo de Provisao (PDD) Mensal:")
spark.sql("""
    SELECT 
        faixa_atraso,
        qtd_faturas,
        FORMAT_NUMBER(valor_total_aberto, 2) as valor_aberto,
        CONCAT(CAST(taxa_pdd * 100 AS INT), '%') as taxa_provisao,
        FORMAT_NUMBER(valor_pdd_calculado, 2) as provisao_necessaria
    FROM tb_gold_resumo_pdd
""").show()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…


 Relatorio Executivo de Provisao (PDD) Mensal:
+---------------+-----------+------------+-------------+-------------------+
|   faixa_atraso|qtd_faturas|valor_aberto|taxa_provisao|provisao_necessaria|
+---------------+-----------+------------+-------------+-------------------+
|    1 a 30 dias|          1|      129.90|           2%|               2.60|
|   31 a 60 dias|          3|      289.70|          10%|              28.97|
|   61 a 90 dias|         22|    2,067.80|          50%|           1,033.90|
|Mais de 90 dias|       1476|  151,052.40|         100%|         151,052.40|
+---------------+-----------+------------+-------------+-------------------+